# 07 - Cruce de hashes entre TODAS las fuentes

Hasta ahora se cruzaron pares puntuales durante el EDA de cada fuente (SilicoData contra las 3 fuentes de TB, Shenzhen/Montgomery contra TBX11K). Incluye también VinDr-CXR (subconjunto de Tuberculosis, 646 imágenes, agregado después de la consolidación inicial). Este notebook hace el cruce completo, **todas las fuentes contra todas**, usando el `phash` (hash perceptual — detecta la misma imagen aunque haya sido reescalada/recomprimida/convertida a otro formato en otra fuente; el `md5` solo detecta copias exactas del mismo archivo).

Incluye el cruce que había quedado pendiente desde el EDA de SilicoData: **Tuberculosis de Chest_x_ray (la fuente base) contra TBX11K y Shenzhen/Montgomery** — el dataset de Rahman et al. de donde sale esa TB es conocido por agregar varias fuentes públicas, así que era un riesgo real, no solo teórico.

**Nota importante sobre el phash**: es una aproximación. Con ~64,000 imágenes en total, es posible que dos radiografías de pacientes distintos, pero visualmente parecidas en composición general, coincidan en el hash sin ser la misma imagen (falso positivo). Por eso este notebook no descarta nada a ciegas: cada grupo de coincidencias se revisa visualmente antes de decidir.

## 0. Configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

sys.path.insert(0, ".")
import eda_utils as eu

HASH_DIR = Path("../Datasets/_eda_hashes")
OUTPUT_DIR = Path("../resultados/eda/_cruce_hashes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42

## 1. Cargar los hashes de las 7 fuentes
Cada CSV tiene columnas ligeramente distintas (`Shenzhen_Montgomery` trae `subsource`, `SilicoData` trae `patient_id`) — `pd.concat` las alinea solo, rellenando con `NaN` donde falte.

**SilicoData**: solo se cargó el hash de Set A (`SilicoData_setA_md5.csv`). Set B se dejó fuera de la consolidación a propósito — en su EDA se encontró que el 81% ya está en Set A, y el resto tiene etiquetas de texto libre bastante más ruidosas; se trata como recurso de anotación aparte, no como fuente de imágenes nuevas para clasificación (si esto se quiere reconsiderar, avisar antes de la consolidación).

In [ ]:
hash_files = sorted(HASH_DIR.glob("*_md5.csv"))
print("Archivos encontrados:")
for f in hash_files:
    print(" ", f.name)

frames = []
for f in hash_files:
    df = pd.read_csv(f)
    frames.append(df)

all_hashes = pd.concat(frames, ignore_index=True)
print()
print(f"Total de imágenes combinadas: {len(all_hashes)}")
all_hashes.groupby("source_dataset").size()

In [ ]:
print("Conteo por fuente y clase:")
all_hashes.groupby(["source_dataset", "target_class"]).size().unstack(fill_value=0)

## 2. Grupos de imágenes con el mismo phash, en más de una fuente
Se agrupa por `phash` sin importar la clase (si el mismo phash aparece con clases distintas en dos fuentes, eso es una alerta seria de conflicto de etiqueta o falso positivo del hash — se reporta aparte, no se resuelve en automático).

In [ ]:
phash_sources = all_hashes.groupby("phash")["source_dataset"].nunique()
cross_source_hashes = phash_sources[phash_sources > 1].index
print(f"Valores de phash que aparecen en más de una fuente: {len(cross_source_hashes)}")

cross_df = all_hashes[all_hashes["phash"].isin(cross_source_hashes)].copy()
print(f"Imágenes involucradas: {len(cross_df)}")

In [ ]:
# Tabla resumen: cuántos grupos duplicados hay por CADA PAR de fuentes
from itertools import combinations

pair_counts = {}
for phash_val, group in cross_df.groupby("phash"):
    sources = sorted(group["source_dataset"].unique())
    for a, b in combinations(sources, 2):
        pair_counts[(a, b)] = pair_counts.get((a, b), 0) + 1

pd.Series(pair_counts).sort_values(ascending=False).rename("grupos_en_comun")

### 2.1 Cruce dirigido: las 5 fuentes de Tuberculosis entre sí

El cruce que había quedado pendiente desde el EDA de SilicoData (Chest_x_ray vs. TBX11K vs. Shenzhen/Montgomery — el dataset de Rahman et al. es conocido por agregar varias fuentes públicas) ahora se extiende a las 5 fuentes de Tuberculosis del proyecto, incluyendo SilicoData y la recién agregada VinDr-CXR.

In [ ]:
tb_sources = ["Chest_x_ray", "TBX11K-simplified", "Shenzhen_Montgomery", "SilicoData", "VinDr_CXR"]
tb_cross = cross_df[
    cross_df["source_dataset"].isin(tb_sources)
    & cross_df["target_class"].isin(["Tuberculosis"])
]
tb_group_sizes = tb_cross.groupby("phash")["source_dataset"].nunique()
tb_real_cross = tb_group_sizes[tb_group_sizes > 1].index
print(f"Grupos de Tuberculosis compartidos entre las 5 fuentes: {len(tb_real_cross)}")
cross_df[cross_df["phash"].isin(tb_real_cross)].sort_values("phash")[
    ["phash", "source_dataset", "target_class", "path"]
]

### 2.2 ¿Hay grupos donde la clase NO coincide entre fuentes?
Esto sería más grave que un simple duplicado: o hay un error de etiqueta en alguna fuente, o el phash dio un falso positivo (dos imágenes distintas que por composición general hashean igual).

In [ ]:
class_conflicts = cross_df.groupby("phash")["target_class"].nunique()
conflict_hashes = class_conflicts[class_conflicts > 1].index
print(f"Grupos con clases distintas entre fuentes: {len(conflict_hashes)}")
if len(conflict_hashes):
    display(cross_df[cross_df["phash"].isin(conflict_hashes)].sort_values("phash")[
        ["phash", "source_dataset", "target_class", "split", "path"]
    ])

## 3. Revisión visual de una muestra de los grupos encontrados
Para confirmar que el phash no está dando falsos positivos antes de descartar nada. Se muestra cada imagen del grupo lado a lado — deberían verse como la misma radiografía.

In [ ]:
def _loader(path):
    if str(path).endswith((".dcm", ".dicom")):
        arr, _ = eu.probe_dicom_normalized(path)
        return arr
    return Image.open(path)

def show_cross_group(group: pd.DataFrame):
    fig, axes = plt.subplots(1, len(group), figsize=(3 * len(group), 3.3))
    axes = np.array(axes).reshape(-1)
    for ax, (_, row) in zip(axes, group.iterrows()):
        try:
            ax.imshow(_loader(row["path"]), cmap="gray")
        except Exception as e:
            ax.text(0.5, 0.5, f"error: {e}", ha="center", va="center")
        ax.set_title(f"{row['source_dataset']}\n{row['target_class']}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

sample_hashes = pd.Series(list(cross_source_hashes)).sample(
    min(8, len(cross_source_hashes)), random_state=RANDOM_SEED
)
for h in sample_hashes:
    print(f"phash {h}:")
    show_cross_group(cross_df[cross_df["phash"] == h])

## 4. Decidir cuál copia se queda: la de mejor resolución
Los CSV de hash no guardaron ancho/alto (para no repetir ese trabajo en cada notebook de EDA) — aquí se vuelve a abrir SOLO las imágenes que están en algún grupo duplicado (son pocas) para medir cuál tiene mayor resolución real.

In [ ]:
def get_dimensions(path):
    try:
        if str(path).endswith((".dcm", ".dicom")):
            arr, err = eu.probe_dicom(path)
            return (None, None) if err else arr.shape[:2]
        with Image.open(path) as im:
            return im.size[1], im.size[0]  # (alto, ancho), consistente con el caso DICOM
    except Exception:
        return (None, None)

dims = [get_dimensions(p) for p in tqdm(cross_df["path"], desc="Midiendo resolución de duplicados")]
cross_df["height"], cross_df["width"] = zip(*dims)
cross_df["area"] = cross_df["height"] * cross_df["width"]

In [ ]:
def pick_keeper(group: pd.DataFrame) -> pd.Series:
    """Se queda la de mayor área (ancho x alto); empate -> se queda la primera por orden estable."""
    return group.sort_values("area", ascending=False).iloc[0]

keepers = cross_df.groupby("phash").apply(pick_keeper)
keeper_paths = set(keepers["path"])
descartar = cross_df[~cross_df["path"].isin(keeper_paths)].copy()
descartar["motivo"] = "duplicado_cruzado_entre_fuentes"

print(f"Imágenes a descartar por ser duplicado de otra fuente (se queda la de mejor resolución): {len(descartar)}")
descartar.groupby(["source_dataset", "target_class"]).size()

In [ ]:
descartar[["path", "phash", "source_dataset", "target_class", "split", "motivo"]].to_csv(
    HASH_DIR / "duplicados_cruzados_a_descartar.csv", index=False
)
print("Guardado en Datasets/_eda_hashes/duplicados_cruzados_a_descartar.csv — el notebook de",
      "consolidación lo usa para excluir estas rutas del pool final.")

## 5. Resumen

In [ ]:
resumen = pd.DataFrame({
    "pool combinado (bruto)": all_hashes.groupby("target_class").size(),
    "a descartar por duplicado cruzado": descartar.groupby("target_class").size(),
}).fillna(0).astype(int)
resumen["pool neto"] = resumen["pool combinado (bruto)"] - resumen["a descartar por duplicado cruzado"]
resumen.to_csv(OUTPUT_DIR / "resumen_cruce_hashes.csv")
resumen

**Resultado real:** el cruce dirigido de la Sección 2.1 entre las 5 fuentes de Tuberculosis (Chest_x_ray, TBX11K, Shenzhen/Montgomery, SilicoData, VinDr-CXR) dio **0 grupos compartidos** — la sospecha sobre Rahman et al. (Chest_x_ray) no se materializó, y VinDr-CXR aporta sus 646 imágenes como completamente nuevas (pool neto de Tuberculosis: 2,646 → 3,292). Tampoco hubo conflictos de clase en ningún grupo cruzado (Sección 2.2: 0).

El solapamiento real detectado fue el ya conocido entre COVID-QU-Ex y RSNA_Pneumonia/Chest_x_ray (17,883 duplicados) más un puñado de casos aislados (39) entre otros pares — 18,151 imágenes en total a descartar, todas en Normal/Pneumonia/COVID-19. Ninguna de las clases de baja representación (Tuberculosis, Silicosis, STB) se ve afectada por el cruce cruzado.